# Bayesian Optimization and HMM/Viterbi Route Reconstruction

This follow-up notebook consumes the feature tables and model-selection record from [01_Data_Preprocessing_and_Feature_Engineering.ipynb](./01_Data_Preprocessing_and_Feature_Engineering.ipynb). Run that notebook first if the prepared feature tables or the initial model comparison are missing.

The selected classifier is Random Forest. A Gaussian-process Bayesian optimizer proposes its hyperparameters using observation-level Top-1 accuracy on the existing 16 held-out validation trips. It never uses the student-test partition. After tuning, the winning configuration is refit on all eligible training-partition trips, then evaluated once on the separate student-test set. The HMM uses RF candidate probabilities as emissions and directed road connectivity plus GPS displacement as transition scores; Viterbi returns the most likely candidate sequence. A one-intermediate-arc connector is inserted when needed to represent a short skipped road segment. Non-connectable transitions remain possible with a penalty so decoding can continue across gaps.

In [1]:
from pathlib import Path
import json
import sys

import joblib
import pandas as pd

working_directory = Path.cwd().resolve()
root_candidates = [working_directory, *working_directory.parents]
root_candidates += [parent / 'PathSnap' for parent in root_candidates]
project_root = next(
    (candidate for candidate in root_candidates
     if (candidate / 'src' / 'models' / 'bayesian_hmm.py').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Could not locate PathSnap/src/models/bayesian_hmm.py from the notebook working directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models.bayesian_hmm import (
    evaluate_hmm_partition,
    optimize_random_forest,
    refit_tuned_random_forest,
)

feature_root = project_root / 'data' / 'processed' / 'features'
cleaned_root = project_root / 'data' / 'processed' / 'cleaned'
previous_results = project_root / 'results' / 'candidate_models'
output_root = previous_results / 'bayesian_hmm'
selection_record = json.loads((previous_results / 'model_selection.json').read_text(encoding='utf-8'))
if selection_record['best_model'] != 'Random Forest':
    raise ValueError(f"This notebook tunes the previously selected Random Forest, not {selection_record['best_model']!r}.")
if selection_record.get('student_test_used_for_selection', False):
    raise ValueError('The prior model-selection record indicates test-set leakage; do not continue.')
output_root.mkdir(parents=True, exist_ok=True)
print(f'Project: {project_root}')
print(f'Chosen model: {selection_record["best_model"]}; validation trips: {selection_record["validation_trip_count"]}; output: {output_root}')

Project: C:\Users\NIkAt\coding\codes\Machine Learning\ML_Hackathon\PathSnap
Chosen model: Random Forest; validation trips: 16; output: C:\Users\NIkAt\coding\codes\Machine Learning\ML_Hackathon\PathSnap\results\candidate_models\bayesian_hmm


## 1. Bayesian Hyperparameter Optimization

The search starts with the previous Random Forest configuration and three space-filling initial trials, then makes four sequential expected-improvement proposals from a Gaussian-process surrogate. All trials use the same deterministic, trip-stratified 200-observation-per-trip sample and the same 64/16 trip split. The objective is validation observation-level Top-1 accuracy; candidate-level metrics are retained for context.

In [2]:
bayesian_results, bayesian_best_params, bayesian_validation_model_path, bayesian_summary = optimize_random_forest(
    feature_root=feature_root,
    output_root=output_root,
    chosen_model_name=selection_record['best_model'],
    observations_per_trip=200,
    initial_points=4,
    bayesian_iterations=4,
    random_seed=42,
)
display(bayesian_results)
print(f'Best parameters: {bayesian_best_params}')
print(f'Validation-only model saved to: {bayesian_validation_model_path}')

Trial 1/8: Top-1=0.8986, F1=0.8772, params={'n_estimators': 50, 'max_depth': 10, 'min_samples_leaf': 10, 'max_features': 0.2425, 'max_samples': 1.0}, fit=16.1s
Trial 2/8: Top-1=0.9000, F1=0.8791, params={'n_estimators': 56, 'max_depth': 8, 'min_samples_leaf': 11, 'max_features': 0.5474, 'max_samples': 0.7557}, fit=29.7s
Trial 3/8: Top-1=0.8971, F1=0.8784, params={'n_estimators': 40, 'max_depth': 11, 'min_samples_leaf': 4, 'max_features': 0.9658, 'max_samples': 0.9475}, fit=49.1s
Trial 4/8: Top-1=0.8991, F1=0.8775, params={'n_estimators': 75, 'max_depth': 14, 'min_samples_leaf': 20, 'max_features': 0.2473, 'max_samples': 0.8316}, fit=21.1s
Trial 5/8: Top-1=0.8998, F1=0.8775, params={'n_estimators': 44, 'max_depth': 8, 'min_samples_leaf': 12, 'max_features': 0.3409, 'max_samples': 0.6971}, fit=12.9s


c:\Users\NIkAt\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\NIkAt\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\NIkAt\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 2 of parameter k1__k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Trial 6/8: Top-1=0.8956, F1=0.8776, params={'n_estimators': 59, 'max_depth': 14, 'min_samples_leaf': 7, 'max_features': 0.9834, 'max_samples': 0.6843}, fit=60.3s


c:\Users\NIkAt\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Trial 7/8: Top-1=0.8991, F1=0.8746, params={'n_estimators': 73, 'max_depth': 6, 'min_samples_leaf': 5, 'max_features': 0.8383, 'max_samples': 0.8127}, fit=50.9s
Trial 8/8: Top-1=0.8992, F1=0.8755, params={'n_estimators': 47, 'max_depth': 6, 'min_samples_leaf': 11, 'max_features': 0.7889, 'max_samples': 0.9901}, fit=35.8s


,trial,n_estimators,max_depth,min_samples_leaf,max_features,max_samples,accuracy,precision,recall,f1,log_loss,observation_top1_accuracy,evaluated_observations,evaluated_candidate_rows,training_seconds,inference_seconds,objective
1,2,56,8,11,0.5474,0.7557,0.992047,0.880509,0.877717,0.879111,0.020850,0.900039,15456,469145,29.723686,0.592784,0.900039
4,5,44,8,12,0.3409,0.6971,0.992054,0.891455,0.864001,0.877513,0.022986,0.899780,15456,469145,12.912974,0.671282,0.899780
7,8,47,6,11,0.7889,0.9901,0.991898,0.886618,0.864648,0.875495,0.021857,0.899198,15456,469145,35.784872,0.570207,0.899198
3,4,75,14,20,0.2473,0.8316,0.991983,0.883317,0.871830,0.877536,0.022018,0.899068,15456,469145,21.059890,0.951016,0.899068
6,7,73,6,5,0.8383,0.8127,0.991851,0.887121,0.862384,0.874578,0.021683,0.899068,15456,469145,50.862485,0.617265,0.899068
0,1,50,10,10,0.2425,1.0000,0.992041,0.892040,0.862836,0.877195,0.022730,0.898551,15456,469145,16.116015,0.672942,0.898551
2,3,40,11,4,0.9658,0.9475,0.991941,0.873266,0.883605,0.878405,0.021558,0.897127,15456,469145,49.101944,0.478246,0.897127
5,6,59,14,7,0.9834,0.6843,0.991896,0.873143,0.882182,0.877639,0.021946,0.895639,15456,469145,60.298315,0.736368,0.895639


Best parameters: {'n_estimators': 56, 'max_depth': 8, 'min_samples_leaf': 11, 'max_features': 0.5474, 'max_samples': 0.7557}
Validation-only model saved to: C:\Users\NIkAt\coding\codes\Machine Learning\ML_Hackathon\PathSnap\results\candidate_models\bayesian_hmm\bayesian_validation_model.joblib


## 2. Refit the Tuned Model on the Training Partition

Once the validation-based hyperparameters are fixed, refit on eligible observations from the complete training partition (including the former validation trips). The separate student-test data is not loaded or used for fitting.

In [3]:
tuned_model_path, tuned_refit_summary = refit_tuned_random_forest(
    feature_root=feature_root,
    output_root=output_root,
    best_params=bayesian_best_params,
    observations_per_trip=500,
    random_seed=42,
)
display(pd.Series(tuned_refit_summary, name='final_refit').to_frame())

,final_refit
model_name,Random Forest
best_params,"{'n_estimators': 56, 'max_depth': 8, 'min_samp..."
training_partition_trip_count,80
contributing_training_trip_count,77
fit_observations,35518
observations_per_trip_cap,500
training_seconds,111.381697
student_test_used_for_training,False
model_path,C:\Users\NIkAt\coding\codes\Machine Learning\M...


## 3. HMM/Viterbi Validation Diagnostics

This diagnostic uses the best Bayesian trial fitted only on the 64 training trips. HMM transition settings are fixed in advance, not tuned on the test data. Compare independent candidate Top-1 predictions with Viterbi-decoded observation accuracy, route continuity, transition feasibility, and expanded-route continuity.

In [4]:
validation_bundle = joblib.load(bayesian_validation_model_path)
validation_hmm_predictions, validation_route_sequences, validation_hmm_metrics = evaluate_hmm_partition(
    feature_root=feature_root,
    cleaned_root=cleaned_root,
    model_bundle=validation_bundle,
    output_root=output_root,
    partition='train',
    split='validation',
)
display(pd.Series(validation_hmm_metrics, name='validation').to_frame())
display(validation_route_sequences.head(20))

,validation
partition,train
split,validation
trip_count,16
scored_trip_count,15
labeled_observations,15456
candidate_rows,908658
candidate_miss_count,0
candidate_miss_rate,0.0
reliable_candidate_references,15456
top1_observation_accuracy,0.900039


,trip_id,route_index,arc_id,inserted_connector,at_observation_id
0,00000001,0,64973,False,0
1,00000001,1,133948,False,6
2,00000001,2,133949,False,7
3,00000001,3,139208,False,8
4,00000001,4,133950,True,10
5,00000001,5,8970,False,10
6,00000001,6,133951,False,14
7,00000001,7,64139,False,17
8,00000001,8,139211,False,19
9,00000001,9,133952,False,21


## 4. Final HMM/Viterbi Evaluation on Student-Test Trips

This is a one-time evaluation of the refitted tuned model. The route-sequence-suspect trip is still decoded for reconstruction output, but its unreliable labels are excluded from scored metrics. Do not select or retune the classifier or transition settings based on these test results.

In [5]:
tuned_bundle = joblib.load(tuned_model_path)
test_hmm_predictions, test_route_sequences, test_hmm_metrics = evaluate_hmm_partition(
    feature_root=feature_root,
    cleaned_root=cleaned_root,
    model_bundle=tuned_bundle,
    output_root=output_root,
    partition='student_test',
)
display(pd.Series(test_hmm_metrics, name='student_test').to_frame())
display(test_route_sequences.head(20))
print(f'Bayesian search results: {output_root / "bayesian_optimization_results.csv"}')
print(f'Student-test Viterbi predictions: {output_root / "student_test_viterbi_route_sequence.csv"}')

,student_test
partition,student_test
split,None
trip_count,10
scored_trip_count,9
labeled_observations,10496
candidate_rows,553240
candidate_miss_count,0
candidate_miss_rate,0.0
reliable_candidate_references,10496
top1_observation_accuracy,0.896246


,trip_id,route_index,arc_id,inserted_connector,at_observation_id
0,00000000,0,17630,False,0
1,00000000,1,17633,False,3
2,00000000,2,211,False,5
3,00000000,3,17638,False,9
4,00000000,4,17645,False,20
5,00000000,5,17659,False,33
6,00000000,6,17674,False,40
7,00000000,7,208,False,58
8,00000000,8,10764,False,68
9,00000000,9,501,False,77


Bayesian search results: C:\Users\NIkAt\coding\codes\Machine Learning\ML_Hackathon\PathSnap\results\candidate_models\bayesian_hmm\bayesian_optimization_results.csv
Student-test Viterbi predictions: C:\Users\NIkAt\coding\codes\Machine Learning\ML_Hackathon\PathSnap\results\candidate_models\bayesian_hmm\student_test_viterbi_route_sequence.csv
